# Phase 2 - Testing

Workflow for testing all modules for phase 2 (agentic workflow). Execute top to bottom

In [1]:
# autoreload modules when executing - makes iterative development easier

%load_ext autoreload 
%autoreload 2

### Query Rewriting
Takes care of formulating proper query for search in vector DB - that includes semantic query formatted properly for this purpose, along with standard keyword filters (price, other features)

Test simple query rewriting - creates optimal semantic query for vector search, along with filter hints for standard fields (price, features)

In [ ]:
from chatshop.config import settings
from chatshop.infra.llm_client import llm_client_for
from chatshop.rag.query_rewriter import QueryRewriter

rewriter_llm = llm_client_for(settings.query_rewriter_model)
query_rewriter = QueryRewriter(rewriter_llm)

In [3]:
query_rewriter.rewrite("Need headphones for commuting, under $100")

RewrittenQuery(semantic_query='noise-cancelling headphones portable long battery life', filter_hints={'max_price': 100, 'min_price': 0, 'min_rating': None, 'extra_filters': {'anc': True, 'use_cases': 'travel'}}, intent_summary='The user is looking for noise-cancelling headphones suitable for commuting that are priced under $100.')

Test a more complex query rewriting, where critical information about headphones is split among multiple messages

In [4]:
from chatshop.rag.query_rewriter import _SYSTEM_PROMPT

messages = [
    {"role": "system", "content": _SYSTEM_PROMPT},
    {"role": "user", "content": "Hey, I need some new headphones."},
    {"role": "assistant", "content": "Sure — what will you mainly use them for?"},
    {"role": "user", "content": "Mostly when I go running."},
    {"role": "assistant", "content": "Got it. Do you have any budget in mind?"},
    {"role": "user", "content": "Yeah, ideally under 120."},
    {"role": "assistant", "content": "Do you prefer earbuds or over-ear?"},
    {"role": "user", "content": "Definitely earbuds. And I hate cables."},
]

query_rewriter.rewrite(messages)

RewrittenQuery(semantic_query='wireless sport earbuds sweat-resistant stable fit', filter_hints={'max_price': 120.0, 'min_price': 0.0, 'min_rating': None, 'extra_filters': {'wireless': True, 'type': 'in-ear', 'use_cases': 'sport'}}, intent_summary='User is looking for wireless sport earbuds suitable for running under $120.')

### Agent workflow - the Planner

The planner is the 'brain', responsible for judging what is next required step. Search for products? Respond to user? Ask for clarifications?

In [ ]:
from chatshop.agent.planner import Planner

planner_llm = llm_client_for(settings.planner_model)
planner = Planner(planner_llm, query_rewriter)

Case 1 - a simple definite prompt

In [6]:
result_searchAction = planner.plan("I need headphones for commuting, under $100")
print(result_searchAction)

SearchAction(action='search', search_plan=SearchPlan(semantic_query='noise-cancelling headphones portable long battery life', filters=SearchFilters(max_price=100, min_price=0, min_rating=None, extra_filters={'anc': True, 'use_cases': 'travel'}), sort_by=None), reasoning_trace="The user's intent is clear: they are looking for headphones specifically for commuting and have a budget of under $100. This allows for a reasonable search to find suitable products that meet these criteria.", intent_summary='The user is seeking affordable noise-cancelling headphones suitable for commuting.')


Case 2 - generic prompt

In [ ]:
result_clarifyAction = planner.plan("Hi, what's up?")
print(result_clarifyAction)

ClarifyAction(action='clarify', question='What specific information or assistance do you need regarding headphones?', reasoning_trace="The user's intent is unclear as they have not provided any specific information or context regarding headphones. I need to ask a focused question to determine what they are looking for.")


Case 3 - vague prompt

In [ ]:
result = planner.plan("Hi, I need god damn heaphones, the best there is")
print(result)

ClarifyAction(action='clarify', question='What specific features or use cases are you looking for in the best headphones?', reasoning_trace="The user's request for 'the best headphones' is vague as it does not specify any particular use case, features, or preferences (e.g., over-ear, noise-canceling, budget). I need to clarify what they mean by 'best' to provide meaningful recommendations.")


In [ ]:
long_conversation = [
    {"role": "user", "content": "Hey, I need some new headphones."},
    {"role": "assistant", "content": "Sure — what will you mainly use them for?"},
    {"role": "user", "content": "Mostly when I go running."},
    {"role": "assistant", "content": "Got it. Do you have any budget in mind?"},
    {"role": "user", "content": "Yeah, ideally under 120."},
    {"role": "assistant", "content": "Do you prefer earbuds or over-ear?"},
    {"role": "user", "content": "Definitely earbuds. And I hate cables."},
]

result = planner.plan(long_conversation)
print(result)

SearchAction(action='search', search_plan=SearchPlan(semantic_query='wireless sport earbuds sweat-resistant stable fit', filters=SearchFilters(max_price=120, min_price=0, min_rating=None, extra_filters={'wireless': True, 'type': 'in-ear', 'use_cases': 'sport'}), sort_by=None), reasoning_trace="The user's intent is now clear: they want wireless earbuds for running, with a budget under $120. This allows for a focused search based on these specific criteria.")


### Hybrid Search

A semantic (vector embedding) + metadata (keywoard) search based on SearchPlan as determined by the Planner. Retrieves best k products matching the SearchPlan.


In [7]:
from re import search

from chatshop.rag.retriever import Retriever
from chatshop.rag.hybrid_search import HybridSearch

retriever = Retriever()
hybrid_search = HybridSearch(retriever)

In [9]:
search_results = hybrid_search.search(result_searchAction.search_plan)
for p in search_results.products:
    print(p.to_context_text())
    print("---")

Audio-Technica Nova 3
Audio-Technica QuietPoint ATH-ANC900BT brings Japanese acoustic precision and reliability to a feature-rich wireless headphone at an accessible price. The 40mm driver system is engineered for the natural, musical sound profile Audio-Technica has perfected over decades in professional studio monitor design. Hybrid Digital Active Noise Cancelling with four microphones produces effective isolation from transportation and environmental noise during commutes. The folding design with premium carrying 
Price: $99.00 | Brand: Audio-Technica | Type: over-ear | Wireless: Yes | ANC: Yes | Battery: 50h | Driver: 40.0mm | Use cases: travel, office
---
Anker Soundcore Life Q35
Anker Soundcore Life Q35 delivers premium features at an unbeatable price. LDAC codec support ensures you're receiving Hi-Res wireless audio with three times more data than standard Bluetooth connections. Three levels of adaptive noise cancellation intelligently analyze your environment and filter distrac

### Evaluator
Evalutes the products based on SearchPlan and the actual search results.
Typical use-case is if user has unreasonable expectations, i.e, he wants all futures but is willing to spend 10 dollars only.
But in more generic way, it should evaluate whether the resulting products are good enough for what the user wants. If not, the planner can decide to e.g. reduce search criteria and offer alternatives.

In [ ]:
from chatshop.agent.evaluator import Evaluator

evaluator_llm = llm_client_for(settings.evaluator_model)
evaluator = Evaluator(evaluator_llm)

Evaluator's verdict will be unsatisfactory in this case:

In [14]:
planner_result = planner.plan("I need headphones for commuting, under $100")
search_results = hybrid_search.search(planner_result.search_plan)

result_eval = evaluator.evaluate(
    intent_summary=planner_result.search_plan.semantic_query,
    constraints=planner_result.search_plan.filters.__dict__, 
    products=search_results.products,
    candidate_count=search_results.candidate_count,
)

print(result_eval)

EvaluatorOutput(satisfactory=False, reason="The results include only two over-ear headphones and three in-ear options, which may not meet the user's preference for portable headphones with long battery life, and the in-ear options have significantly shorter battery life compared to the over-ear options.")


We feed it into the planner on next iteration, do hybrid search and we will pass evaluator this time:

In [15]:
planner_next_iteration = planner.plan("I need headphones for commuting, under $100", previous_results=search_results.products, evaluator_feedback=result_eval.verdict())

search_results_next_iteration = hybrid_search.search(planner_next_iteration.search_plan)

result_eval_next_iteration = evaluator.evaluate(
    intent_summary=planner_next_iteration.search_plan.semantic_query,
    constraints=planner_next_iteration.search_plan.filters.__dict__, 
    products=search_results_next_iteration.products,
    candidate_count=search_results_next_iteration.candidate_count,
)

print(result_eval_next_iteration)

EvaluatorOutput(satisfactory=True, reason="All retrieved products meet the user's intent for noise-cancelling, portable, over-ear headphones with long battery life, and they all fall within the specified price range.")


In [16]:
print(search_results_next_iteration)

SearchResult(products=[Product(product_id='67', title='Audio-Technica Nova 3', description='Audio-Technica QuietPoint ATH-ANC900BT brings Japanese acoustic precision and reliability to a feature-rich wireless headphone at an accessible price. The 40mm driver system is engineered for the natural, musical sound profile Audio-Technica has perfected over decades in professional studio monitor design. Hybrid Digital Active Noise Cancelling with four microphones produces effective isolation from transportation and environmental noise during commutes. The folding design with premium carrying ', price=99.0, brand='Audio-Technica', type='over-ear', wireless=True, anc=True, battery_hours=50, waterproof_rating=None, driver_size_mm=40.0, use_cases=['travel', 'office']), Product(product_id='7', title='Anker Soundcore Life Q35', description="Anker Soundcore Life Q35 delivers premium features at an unbeatable price. LDAC codec support ensures you're receiving Hi-Res wireless audio with three times mo

## Agent Loop


In [ ]:
from chatshop.config import settings
from chatshop.infra.llm_client import llm_client_for
from chatshop.agent.agent_loop import AgentLoop
from chatshop.agent.evaluator import Evaluator
from chatshop.agent.planner import Planner
from chatshop.rag.query_rewriter import QueryRewriter
from chatshop.rag.retriever import Retriever
from chatshop.rag.hybrid_search import HybridSearch

planner_llm   = llm_client_for(settings.planner_model)
rewriter_llm  = llm_client_for(settings.query_rewriter_model)
evaluator_llm = llm_client_for(settings.evaluator_model)
synthesis_llm = llm_client_for(settings.synthesis_model)

query_rewriter = QueryRewriter(rewriter_llm)
planner        = Planner(planner_llm, query_rewriter)
evaluator      = Evaluator(evaluator_llm)
hybrid_search  = HybridSearch(Retriever())

agent = AgentLoop(planner, evaluator, hybrid_search, synthesis_llm, max_iterations=5)

In [12]:
print(agent.run("I need wireless earbuds for running under $100", history=[]))

I recommend the **Jabra Endurance Peak 3** as the top choice for wireless earbuds for running. They are specifically designed for athletes with a secure fit, IP57 waterproof rating, and 9 hours of battery life, making them perfect for outdoor training regardless of the weather.

Here are some alternatives:

1. **JLab Epic Air Sport ANC Gen 2**
   - Price: $79.00
   - Rating: Not specified
   - Battery: 15 hours
   - ANC: Yes
   - Waterproof: IPX5
   - Ideal for: Sport, travel, with active noise cancellation for focused workouts.

2. **Pioneer SE-C5TW**
   - Price: $69.00
   - Rating: Not specified
   - Battery: 9 hours
   - ANC: No
   - Waterproof: IPX5
   - Ideal for: Sport, travel, offers a stable fit and detailed sound.

3. **1More True Wireless X**
   - Price: $69.00
   - Rating: Not specified
   - Battery: 7 hours
   - ANC: No
   - Waterproof: IPX5
   - Ideal for: Sport, travel, known for sound quality and comfort.

4. **JBL Tune Buds**
   - Price: $49.00
   - Rating: Not specifie